In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import (
	AutoTokenizer,
	AutoModelForSequenceClassification,
	get_linear_schedule_with_warmup,
)
from tqdm import tqdm
import re

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [2]:
train_df = pd.read_csv('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256/data1/train.csv')
test_df  = pd.read_csv('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256/data1/test.csv')

print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(train_df['label'].value_counts().sort_index())
train_df.head(3)

Train: 252000 rows | Test: 168000 rows
label
0    50400
1    50400
2    50400
3    50400
4    50400
Name: count, dtype: int64


,id,sentence,label
0,0,Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...,4
1,1,Passt wie angegossen\n\nTasche kam schnell und...,4
2,2,Schlechte Qualität\n\nIch habe sehr lange auf ...,1


In [3]:
def clean_text(text):
	text = re.sub(r'\s+', ' ', text).strip()
	return text

train_df['sentence'] = train_df['sentence'].apply(clean_text)
test_df['sentence']  = test_df['sentence'].apply(clean_text)

In [4]:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(
	train_df['sentence'].tolist(),
	train_df['label'].tolist(),
	test_size=0.10,
	random_state=SEED,
	stratify=train_df['label'].tolist()
)
print(f'Train split: {len(train_sentences)} | Val split: {len(val_sentences)}')

Train split: 226800 | Val split: 25200


In [5]:
class ReviewsDataset(Dataset):
	def __init__(self, encodings, labels):
		self.encodings = encodings
		self.labels = labels

	def __getitem__(self, idx):
		item = {key: val[idx].clone() for key, val in self.encodings.items()}
		item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
		return item

	def __len__(self):
		return len(self.labels)

In [6]:
BERT_MODEL_NAME = 'nlptown/bert-base-multilingual-uncased-sentiment'
BERT_MODEL_PATH = '/kaggle/input/models/vladcvx/cil-big-bert/pytorch/default/1/best_model_init_num_epochs-5lr-1e-06step_size-Nonegamma-Nonetrain_size-180000val_size-10000.pth'

# bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

# if not os.path.exists('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256/data1/bert-encodings-256/bert_val_encodings.pt'):
# 	bert_val_enc  = bert_tokenizer(val_sentences, truncation=True, padding=True, max_length=256, return_tensors='pt')
# 	bert_test_enc = bert_tokenizer(test_df['sentence'].tolist(), truncation=True, padding=True, max_length=256, return_tensors='pt')
# 	torch.save(bert_val_enc,  '/kaggle/working/bert-encodings/bert_val_encodings.pt')
# 	torch.save(bert_test_enc, '/kaggle/working/bert-encodings/bert_test_encodings.pt')
# else:
# 	bert_val_enc  = torch.load('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256/data1/bert-encodings-256/bert_val_encodings.pt',  weights_only=False)
# 	bert_test_enc = torch.load('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256/data1/bert-encodings-256/bert_test_encodings.pt', weights_only=False)

# bert_val_dataset  = ReviewsDataset(bert_val_enc,  val_labels)
# bert_test_dataset = ReviewsDataset(bert_test_enc, [-1] * len(test_df))

# bert_val_loader  = DataLoader(bert_val_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
# bert_test_loader = DataLoader(bert_test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=5)
bert_model.load_state_dict(torch.load(BERT_MODEL_PATH, map_location=device))
bert_model = bert_model.to(device)
# bert_model = torch.nn.DataParallel(bert_model, device_ids=[0,1]).to(device)
bert_model.eval()
print('BERT model loaded.')

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BERT model loaded.


In [7]:
ROBERTA_MODEL_NAME = 'xlm-roberta-large'
ROBERTA_MODEL_PATH = '/kaggle/input/models/vladcvx/cil-roberta-1-epoch/pytorch/default/1/roberta_best_model_init_num_epochs-5lr-1e-06step_size-Nonegamma-Nonetrain_size-226800val_size-25200.pth'

# roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME)

# if not os.path.exists('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-roberta/data1/roberta-encodings/roberta_val_encodings.pt'):
# 	roberta_val_enc  = roberta_tokenizer(val_sentences, truncation=True, padding=True, max_length=256, return_tensors='pt')
# 	roberta_test_enc = roberta_tokenizer(test_df['sentence'].tolist(), truncation=True, padding=True, max_length=256, return_tensors='pt')
# 	torch.save(roberta_val_enc,  '/kaggle/working/roberta-encodings/roberta_val_encodings.pt')
# 	torch.save(roberta_test_enc, '/kaggle/working/roberta-encodings/roberta_test_encodings.pt')
# else:
# 	roberta_val_enc  = torch.load('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-roberta/data1/roberta-encodings/roberta_val_encodings.pt',  weights_only=False)
# 	roberta_test_enc = torch.load('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-roberta/data1/roberta-encodings/roberta_test_encodings.pt', weights_only=False)

# roberta_val_dataset  = ReviewsDataset(roberta_val_enc,  val_labels)
# roberta_test_dataset = ReviewsDataset(roberta_test_enc, [-1] * len(test_df))

# roberta_val_loader  = DataLoader(roberta_val_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
# roberta_test_loader = DataLoader(roberta_test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL_NAME, num_labels=5)
roberta_model.load_state_dict(torch.load(ROBERTA_MODEL_PATH, map_location=device))
roberta_model = roberta_model.to(device)
# roberta_model = torch.nn.DataParallel(roberta_model, device_ids=[0,1]).to(device)
roberta_model.eval()
print('RoBERTa model loaded.')

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa model loaded.


In [8]:
ELECTRA_MODEL_NAME = 'google/electra-base-discriminator'
ELECTRA_MODEL_PATH = '/kaggle/input/models/vladcvx/cil-electra-5-epochs/pytorch/default/1/best_model_init_num_epochs-5_lr-1e-06_step_size-None_gamma-None_train_size-226800_val_size-25200.pth'

# electra_tokenizer = AutoTokenizer.from_pretrained(ELECTRA_MODEL_NAME)

# if not os.path.exists('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-electra/electra_val_encodings.pt'):
# 	electra_val_enc  = electra_tokenizer(val_sentences, truncation=True, padding=True, max_length=256, return_tensors='pt')
# 	electra_test_enc = electra_tokenizer(test_df['sentence'].tolist(), truncation=True, padding=True, max_length=256, return_tensors='pt')
# 	torch.save(electra_val_enc,  '/kaggle/working/electra-encodings/electra_val_encodings.pt')
# 	torch.save(electra_test_enc, '/kaggle/working/electra-encodings/electra_test_encodings.pt')
# else:
# 	electra_val_enc  = torch.load('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-electra/electra_val_encodings.pt',  weights_only=False)
# 	electra_test_enc = torch.load('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-electra/electra_test_encodings.pt', weights_only=False)

# electra_val_dataset  = ReviewsDataset(electra_val_enc,  val_labels)
# electra_test_dataset = ReviewsDataset(electra_test_enc, [-1] * len(test_df))

# electra_val_loader  = DataLoader(electra_val_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
# electra_test_loader = DataLoader(electra_test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

electra_model = AutoModelForSequenceClassification.from_pretrained(ELECTRA_MODEL_NAME, num_labels=5)
electra_model.load_state_dict(torch.load(ELECTRA_MODEL_PATH, map_location=device))
electra_model = electra_model.to(device)
# electra_model = torch.nn.DataParallel(electra_model, device_ids=[0,1]).to(device)
electra_model.eval()
print('Electra model loaded.')

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Electra model loaded.


In [9]:
DISTILBERT_MODEL_NAME = 'distilbert-base-multilingual-cased'
DISTILBERT_MODEL_PATH = '/kaggle/input/models/vladcvx/cil-distilbert-3-epochs/pytorch/default/1/best_model_init_num_epochs-5lr-1e-05step_size-Nonegamma-Nonetrain_size-226800val_size-25200.pth'

# distilbert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_MODEL_NAME)

# if not os.path.exists('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-distilbert/data1/distilbert-encodings/distilbert_val_encodings.pt'):
# 	distilbert_val_enc  = distilbert_tokenizer(val_sentences, truncation=True, padding=True, max_length=256, return_tensors='pt')
# 	distilbert_test_enc = distilbert_tokenizer(test_df['sentence'].tolist(), truncation=True, padding=True, max_length=256, return_tensors='pt')
# 	torch.save(distilbert_val_enc,  '/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-distilbert/data1/distilbert-encodings/distilbert_val_encodings.pt')
# 	torch.save(distilbert_test_enc, '/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-distilbert/data1/distilbert-encodings/distilbert_test_encodings.pt')
# else:
# 	distilbert_val_enc  = torch.load('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-distilbert/data1/distilbert-encodings/distilbert_val_encodings.pt',  weights_only=False)
# 	distilbert_test_enc = torch.load('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-distilbert/data1/distilbert-encodings/distilbert_test_encodings.pt', weights_only=False)

# distilbert_val_dataset  = ReviewsDataset(distilbert_val_enc,  val_labels)
# distilbert_test_dataset = ReviewsDataset(distilbert_test_enc, [-1] * len(test_df))

# distilbert_val_loader  = DataLoader(distilbert_val_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
# distilbert_test_loader = DataLoader(distilbert_test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(DISTILBERT_MODEL_NAME, num_labels=5)
distilbert_model.load_state_dict(torch.load(DISTILBERT_MODEL_PATH, map_location=device))
distilbert_model = distilbert_model.to(device)
# distilbert_model = torch.nn.DataParallel(distilbert_model, device_ids=[0,1]).to(device)
distilbert_model.eval()
print('Distilbert model loaded.')

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Distilbert model loaded.


In [10]:
def predict_probs(model, dataloader):
	model.eval()
	all_probs = []
	with torch.no_grad():
		for batch in tqdm(dataloader):
			batch = {k: v.to(device) for k, v in batch.items()}
			batch.pop('labels')
			outputs = model(**batch)
			probs = F.softmax(outputs.logits, dim=1)
			all_probs.append(probs.cpu().numpy())
	return np.vstack(all_probs) #(N, 5)


# bert_probs_val  = predict_probs(bert_model, bert_val_loader)
# roberta_probs_val = predict_probs(roberta_model, roberta_val_loader)
# electra_probs_val  = predict_probs(electra_model, electra_val_loader)


# bert_probs_test   = predict_probs(bert_model, bert_test_loader)
# roberta_probs_test = predict_probs(roberta_model, roberta_test_loader)
# electra_probs_test   = predict_probs(electra_model, electra_test_loader)


bert_probs_val = np.load('/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs/bert_probs_val.npy')
roberta_probs_val = np.load('/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs/roberta_probs_val.npy')
bert_probs_test = np.load('/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs/bert_probs_test.npy')
roberta_probs_test = np.load('/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs/roberta_probs_test.npy')
electra_probs_val = np.load('/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs/electra_probs_val.npy')
electra_probs_test = np.load('/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs/electra_probs_test.npy')
distilbert_probs_val  = np.load('/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs/distilbert_probs_val.npy')
distilbert_probs_test = np.load('/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs/distilbert_probs_test.npy')


print(f'bert_probs_val shape:    {bert_probs_val.shape}')
print(f'roberta_probs_val shape: {roberta_probs_val.shape}')
print(f'electra_probs_val shape: {electra_probs_val.shape}')
print(f'distilbert_probs_val shape: {distilbert_probs_val.shape}')

bert_probs_val shape:    (25200, 5)
roberta_probs_val shape: (25200, 5)
electra_probs_val shape: (25200, 5)
distilbert_probs_val shape: (25200, 5)


In [11]:
# np.save('/kaggle/working/bert_probs_test.npy', bert_probs_test)
# np.save('/kaggle/working/roberta_probs_test.npy', roberta_probs_test)
# np.save('/kaggle/working/bert_probs_val.npy', bert_probs_val)
# np.save('/kaggle/working/roberta_probs_val.npy', roberta_probs_val)
# np.save('/kaggle/working/electra_probs_val.npy', electra_probs_val)
# np.save('/kaggle/working/electra_probs_test.npy', electra_probs_test)

In [ ]:
class StackingMLP(nn.Module):
	def __init__(self, input_dim=15, hidden_dim=64, num_classes=5):
		super().__init__()
		self.mlp = nn.Sequential(
			nn.Linear(input_dim, hidden_dim),
			nn.ReLU(),
			nn.Dropout(0.1),
			nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim//2, hidden_dim//4),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim//4, num_classes),
		)

	def forward(self, x):
		return self.mlp(x)

In [ ]:
from sklearn.model_selection import KFold

X = np.hstack([bert_probs_val, roberta_probs_val, electra_probs_val, distilbert_probs_val])
y = np.array(val_labels)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds = np.zeros((len(X), 5))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_vl = X[train_idx], X[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    
    model_fold = StackingMLP(input_dim=20, hidden_dim=2048, num_classes=5).to(device)
    opt = torch.optim.Adam(model_fold.parameters(), lr=1e-3)
    
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    y_tr_t  = torch.tensor(y_tr, dtype=torch.long).to(device)
    X_vl_t  = torch.tensor(X_vl, dtype=torch.float32).to(device)
    
    for epoch in tqdm(range(300)):
        model_fold.train()
        opt.zero_grad()
        loss = nn.CrossEntropyLoss()(model_fold(X_tr_t), y_tr_t)
        loss.backward()
        opt.step()
    
    model_fold.eval()
    with torch.no_grad():
        oof_preds[val_idx] = F.softmax(model_fold(X_vl_t), dim=1).cpu().numpy()
    
    fold_score = 1.0 - np.mean(np.abs(np.argmax(oof_preds[val_idx], axis=1) - y_vl)) / 4.0
    print(f'Fold {fold+1} score: {fold_score:.4f}')

oof_score = 1.0 - np.mean(np.abs(np.argmax(oof_preds, axis=1) - y)) / 4.0
print(f'\nOOF score: {oof_score:.4f}')

'''
Fold 1 score: 0.9069
Fold 2 score: 0.9073
Fold 3 score: 0.9077
Fold 4 score: 0.9046
Fold 5 score: 0.9094

OOF score: 0.9072 - def hidden_dim=64

'''

100%|██████████| 300/300 [17:00<00:00,  3.40s/it]


Fold 1 score: 0.9049


  0%|          | 1/300 [00:07<35:49,  7.19s/it]


KeyboardInterrupt: 

In [27]:
for name, p in [('bert', bert_probs_val), ('roberta', roberta_probs_val), 
                ('electra', electra_probs_val), ('distilbert', distilbert_probs_val)]:
    preds = np.argmax(p, axis=1)
    score = 1 - np.mean(np.abs(preds - y_val))/4
    print(f"{name}: {score:.4f}")

bert: 0.9007
roberta: 0.9054
electra: 0.8946
distilbert: 0.8947


In [19]:
X_val = np.hstack([bert_probs_val, roberta_probs_val, electra_probs_val, distilbert_probs_val])
y_val = np.array(val_labels)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)

hyperparams_stack = {
	'num_epochs': 500,
	'lr': 1e-3,
	'hidden_dim': 4096,
    'input_dim': 20,
    'val_len': 25200,
}
hyperparams_str = "%".join(f"{k}-{v}" for k, v in hyperparams_stack.items())

stacking_model = StackingMLP(input_dim=hyperparams_stack['input_dim'], hidden_dim=hyperparams_stack['hidden_dim'], num_classes=5).to(device)
optimizer_stack = torch.optim.Adam(stacking_model.parameters(), lr=hyperparams_stack['lr'])
criterion = nn.CrossEntropyLoss()
scheduler_stack = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_stack, T_max=500, eta_min=1e-6
)

best_stack_score = 0.0

for epoch in range(hyperparams_stack['num_epochs']):
	stacking_model.train()
	optimizer_stack.zero_grad()
	logits = stacking_model(X_val_tensor)
	loss = criterion(logits, y_val_tensor)
	loss.backward()
	optimizer_stack.step()
	scheduler_stack.step()
    

	stacking_model.eval()
	with torch.no_grad():
		preds = torch.argmax(stacking_model(X_val_tensor), dim=1).cpu().numpy()
	mae = np.mean(np.abs(preds - y_val))
	score = 1.0 - mae / 4.0

	if score > best_stack_score:
		best_stack_score = score
		torch.save(stacking_model.state_dict(), f'/kaggle/working/best_stacking_mlp_{hyperparams_str}.pth')

	print(f'Epoch {epoch+1:3d} | loss={loss.item():.4f} | val_score={score:.4f}')

print(f'\nBest stacking val score: {best_stack_score:.4f}')

Epoch   1 | loss=1.6107 | val_score=0.8955
Epoch   2 | loss=1.1855 | val_score=0.9035
Epoch   3 | loss=0.8509 | val_score=0.8524
Epoch   4 | loss=1.5064 | val_score=0.8971
Epoch   5 | loss=1.0188 | val_score=0.8826
Epoch   6 | loss=1.2224 | val_score=0.8935
Epoch   7 | loss=1.0515 | val_score=0.9013
Epoch   8 | loss=0.8795 | val_score=0.9051
Epoch   9 | loss=0.8364 | val_score=0.9041
Epoch  10 | loss=0.8778 | val_score=0.9010
Epoch  11 | loss=0.9176 | val_score=0.9009
Epoch  12 | loss=0.9119 | val_score=0.9035
Epoch  13 | loss=0.8747 | val_score=0.9060
Epoch  14 | loss=0.8344 | val_score=0.9066
Epoch  15 | loss=0.8101 | val_score=0.9061
Epoch  16 | loss=0.8064 | val_score=0.9051
Epoch  17 | loss=0.8165 | val_score=0.9041
Epoch  18 | loss=0.8245 | val_score=0.9047
Epoch  19 | loss=0.8241 | val_score=0.9054
Epoch  20 | loss=0.8159 | val_score=0.9063
Epoch  21 | loss=0.8052 | val_score=0.9066
Epoch  22 | loss=0.7962 | val_score=0.9070
Epoch  23 | loss=0.7917 | val_score=0.9072
Epoch  24 |

KeyboardInterrupt: 

In [ ]:
X_val = np.hstack([bert_probs_val, roberta_probs_val, electra_probs_val, distilbert_probs_val])
y_val = np.array(val_labels)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)

hyperparams_stack = {
	'num_epochs': 500,
	'lr': 1e-3,
	'hidden_dim': 2048,
    'input_dim': 20,
    'val_len': 25200,
}
hyperparams_str = "%".join(f"{k}-{v}" for k, v in hyperparams_stack.items())

stacking_model = StackingMLP(input_dim=hyperparams_stack['input_dim'], num_classes=5).to(device)
optimizer_stack = torch.optim.Adam(stacking_model.parameters(), lr=hyperparams_stack['lr'])
criterion = nn.CrossEntropyLoss()
scheduler_stack = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_stack, T_max=500, eta_min=1e-6
)

best_stack_score = 0.0

for epoch in range(hyperparams_stack['num_epochs']):
	stacking_model.train()
	optimizer_stack.zero_grad()
	logits = stacking_model(X_val_tensor)
	loss = criterion(logits, y_val_tensor)
	loss.backward()
	optimizer_stack.step()
	scheduler_stack.step()
    

	stacking_model.eval()
	with torch.no_grad():
		preds = torch.argmax(stacking_model(X_val_tensor), dim=1).cpu().numpy()
	mae = np.mean(np.abs(preds - y_val))
	score = 1.0 - mae / 4.0

	if score > best_stack_score:
		best_stack_score = score
		torch.save(stacking_model.state_dict(), f'/kaggle/working/best_stacking_mlp_{hyperparams_str}.pth')

	print(f'Epoch {epoch+1:3d} | loss={loss.item():.4f} | val_score={score:.4f}')

print(f'\nBest stacking val score: {best_stack_score:.4f}')

Epoch   1 | loss=1.6114 | val_score=0.5000
Epoch   2 | loss=1.6095 | val_score=0.5000
Epoch   3 | loss=1.6077 | val_score=0.5000
Epoch   4 | loss=1.6056 | val_score=0.5000
Epoch   5 | loss=1.6036 | val_score=0.5000
Epoch   6 | loss=1.6015 | val_score=0.5009
Epoch   7 | loss=1.5993 | val_score=0.5057
Epoch   8 | loss=1.5968 | val_score=0.5164
Epoch   9 | loss=1.5943 | val_score=0.5375
Epoch  10 | loss=1.5917 | val_score=0.5598
Epoch  11 | loss=1.5889 | val_score=0.5715
Epoch  12 | loss=1.5861 | val_score=0.5785
Epoch  13 | loss=1.5831 | val_score=0.5824
Epoch  14 | loss=1.5802 | val_score=0.5859
Epoch  15 | loss=1.5771 | val_score=0.5886
Epoch  16 | loss=1.5738 | val_score=0.5908
Epoch  17 | loss=1.5703 | val_score=0.5931
Epoch  18 | loss=1.5663 | val_score=0.5951
Epoch  19 | loss=1.5634 | val_score=0.5969
Epoch  20 | loss=1.5592 | val_score=0.5985
Epoch  21 | loss=1.5550 | val_score=0.6001
Epoch  22 | loss=1.5511 | val_score=0.6412
Epoch  23 | loss=1.5466 | val_score=0.6785
Epoch  24 |

In [16]:
stacking_model.load_state_dict(torch.load('/kaggle/working/best_stacking_mlp_num_epochs-500%lr-0.001%hidden_dim-2048%input_dim-20%val_len-25200.pth', map_location=device))
stacking_model.eval()

X_test = np.hstack([bert_probs_test, roberta_probs_test, electra_probs_test, distilbert_probs_test])  # (N_test, 20)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

with torch.no_grad():
	final_preds = torch.argmax(stacking_model(X_test_tensor), dim=1).cpu().numpy()

submission_df = pd.DataFrame({
	'id'   : test_df['id'].astype(int),
	'label': final_preds.astype(int)
})

assert len(submission_df) == len(test_df)
assert submission_df['label'].isin([0,1,2,3,4]).all()

submission_df.to_csv('/kaggle/working/submission_stacking.csv', index=False)
print('Submission saved.')
print(submission_df.head(5))

Submission saved.
   id  label
0   0      0
1   1      1
2   2      3
3   3      1
4   4      2


In [ ]:
best = 0.0
for w1 in np.arange(0.0, 0.9, 0.05):
    for w2 in np.arange(0.0, 0.9 - w1, 0.05):
        w3 = 1 - w1 - w2
        avg_probs = w1 * bert_probs_val + w2 * roberta_probs_val + w3 * electra_probs_val
        preds = np.argmax(avg_probs, axis=1)
        score = 1 - np.mean(np.abs(preds - y_val)) / 4
        if score > best:
            best = score
            best_w = (w1, w2, w3)
print(f"Best weighted avg: {best:.4f}, weights: {best_w}")

In [17]:
best = 0.0
best_w = None

for w1 in np.arange(0.0, 1.01, 0.01):
    for w2 in np.arange(0.0, 1.01 - w1, 0.01):
        for w3 in np.arange(0.0, 1.01 - w1 - w2, 0.01):
                w4 = 1 - w1 - w2 - w3
                if w4 < -0.01:
                    continue
                avg = w1*bert_probs_val + w2*roberta_probs_val + w3*electra_probs_val + w4*distilbert_probs_val
                preds = np.argmax(avg, axis=1)
                score = 1 - np.mean(np.abs(preds - y_val)) / 4
                if score > best:
                    best = score
                    best_w = (w1, w2, w3, w4)

print(f"Best weighted avg: {best:.4f}")
print(f"bert_ft={best_w[0]:.2f}, roberta={best_w[1]:.2f}, electra={best_w[2]:.2f}, distilbert={best_w[3]:.2f}")

Best weighted avg: 0.9076


IndexError: tuple index out of range

In [18]:
print(f"bert_ft={best_w[0]:.2f}, roberta={best_w[1]:.2f}, electra={best_w[2]:.2f}, distilbert={best_w[3]:.2f}")

bert_ft=0.19, roberta=0.49, electra=0.15, distilbert=0.17


In [ ]:
from sklearn.model_selection import train_test_split

idx_tr, idx_ho = train_test_split(np.arange(len(y_val)), test_size=0.3, random_state=42, stratify=y_val)

bert_tr, bert_ho = bert_probs_val[idx_tr], bert_probs_val[idx_ho]
roberta_tr, roberta_ho = roberta_probs_val[idx_tr], roberta_probs_val[idx_ho]
electra_tr, electra_ho = electra_probs_val[idx_tr], electra_probs_val[idx_ho]
distilbert_tr, distilbert_ho = distilbert_probs_val[idx_tr], distilbert_probs_val[idx_ho]
y_tr, y_ho = y_val[idx_tr], y_val[idx_ho]

In [ ]:
best = 0.0
best_w = None

for w1 in np.arange(0.0, 1.01, 0.05):
    for w2 in np.arange(0.0, 1.01 - w1, 0.05):
        for w3 in np.arange(0.0, 1.01 - w1 - w2, 0.05):
            w4 = 1 - w1 - w2 - w3
            if w4 < -0.01: continue
            avg = w1*bert_tr + w2*roberta_tr + w3*electra_tr + w4*distilbert_tr
            preds = np.argmax(avg, axis=1)
            score = 1 - np.mean(np.abs(preds - y_tr)) / 4
            if score > best:
                best = score
                best_w = (w1, w2, w3, w4)

print(f"Tuning score: {best:.4f}, weights: {best_w}")

w1, w2, w3, w4 = best_w
avg_ho = w1*bert_ho + w2*roberta_ho + w3*electra_ho + w4*distilbert_ho
preds_ho = np.argmax(avg_ho, axis=1)
honest_score = 1 - np.mean(np.abs(preds_ho - y_ho)) / 4
print(f"HONEST holdout score: {honest_score:.4f}")

Tuning score: 0.9075, weights: (np.float64(0.2), np.float64(0.45), np.float64(0.15000000000000002), np.float64(0.2))
HONEST holdout score: 0.9074


In [31]:
w1, w2, w3, w4 = 0.20, 0.45, 0.15, 0.20
avg_test = w1*bert_probs_test + w2*roberta_probs_test + w3*electra_probs_test + w4*distilbert_probs_test
final_preds = np.argmax(avg_test, axis=1)

submission_df = pd.DataFrame({
    'id': test_df['id'].astype(int),
    'label': final_preds.astype(int)
})
submission_df.to_csv('/kaggle/working/submission_weighted_avg_4models.csv', index=False)

In [ ]:
from sklearn.model_selection import train_test_split

idx_tr, idx_ho = train_test_split(np.arange(len(y_val)), test_size=0.3, random_state=42, stratify=y_val)

X_full = np.hstack([bert_probs_val, roberta_probs_val, electra_probs_val, distilbert_probs_val])
X_tr, X_ho = X_full[idx_tr], X_full[idx_ho]
y_tr, y_ho = y_val[idx_tr], y_val[idx_ho]

X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
y_tr_t = torch.tensor(y_tr, dtype=torch.long).to(device)
X_ho_t = torch.tensor(X_ho, dtype=torch.float32).to(device)

class StackingSmallMLP(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=32, num_classes=5):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x):
        return self.mlp(x)

model = StackingSmallMLP(input_dim=20, hidden_dim=1024).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

best_ho_score = 0.0
best_state = None
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_tr_t), y_tr_t)
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        preds_ho = torch.argmax(model(X_ho_t), dim=1).cpu().numpy()
    score_ho = 1 - np.mean(np.abs(preds_ho - y_ho)) / 4
    
    if score_ho > best_ho_score:
        best_ho_score = score_ho
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print('Epoch: ', epoch, ' - Current score: ', score_ho)

print(f"Best HONEST holdout MLP score: {best_ho_score:.4f}")
model.load_state_dict(best_state)

X_test = np.hstack([bert_probs_test, roberta_probs_test, electra_probs_test, distilbert_probs_test])
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
model.eval()
with torch.no_grad():
    final_preds = torch.argmax(model(X_test_t), dim=1).cpu().numpy()

submission_df = pd.DataFrame({'id': test_df['id'].astype(int), 'label': final_preds.astype(int)})
submission_df.to_csv('/kaggle/working/submission_mlp_honest.csv', index=False)

Epoch:  0  - Current score:  0.8830026455026455
Epoch:  1  - Current score:  0.901818783068783
Epoch:  2  - Current score:  0.9034722222222222
Epoch:  3  - Current score:  0.9049933862433862
Epoch:  4  - Current score:  0.9047949735449735
Epoch:  5  - Current score:  0.9046626984126984
Epoch:  6  - Current score:  0.904728835978836
Epoch:  7  - Current score:  0.9051256613756614
Epoch:  8  - Current score:  0.9052579365079365
Epoch:  9  - Current score:  0.9052910052910053
Epoch:  10  - Current score:  0.9053240740740741
Epoch:  11  - Current score:  0.9053240740740741
Epoch:  12  - Current score:  0.9053902116402116
Epoch:  13  - Current score:  0.9054232804232805
Epoch:  14  - Current score:  0.9052910052910053
Epoch:  15  - Current score:  0.9055224867724868
Epoch:  16  - Current score:  0.9055555555555556
Epoch:  17  - Current score:  0.9057208994708995
Epoch:  18  - Current score:  0.9057539682539683
Epoch:  19  - Current score:  0.9055555555555556
Epoch:  20  - Current score:  0.

In [ ]:
from sklearn.model_selection import train_test_split

idx_tr, idx_ho = train_test_split(np.arange(len(y_val)), test_size=0.3, random_state=42, stratify=y_val)

X_full = np.hstack([bert_probs_val, roberta_probs_val, electra_probs_val, distilbert_probs_val])
X_tr, X_ho = X_full[idx_tr], X_full[idx_ho]
y_tr, y_ho = y_val[idx_tr], y_val[idx_ho]

X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
y_tr_t = torch.tensor(y_tr, dtype=torch.long).to(device)
X_ho_t = torch.tensor(X_ho, dtype=torch.float32).to(device)

class StackingSmallMLP(nn.Module):
    def __init__(self, input_dim=15, hidden_dim=64, num_classes=5):
        super().__init__()
        self.mlp = nn.Sequential(
    		nn.Linear(input_dim, hidden_dim),
    		nn.ReLU(),
    		nn.Dropout(0.1),
    		nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim//2, hidden_dim//4),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim//4, num_classes),
    	)
    
    def forward(self, x):
        return self.mlp(x)

model = StackingSmallMLP(input_dim=20, hidden_dim=2048).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

best_ho_score = 0.0
best_state = None
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_tr_t), y_tr_t)
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        preds_ho = torch.argmax(model(X_ho_t), dim=1).cpu().numpy()
    score_ho = 1 - np.mean(np.abs(preds_ho - y_ho)) / 4
    
    if score_ho > best_ho_score:
        best_ho_score = score_ho
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print('Epoch: ', epoch, ' - Current score: ', score_ho)

print(f"Best HONEST holdout MLP score: {best_ho_score:.4f}")
model.load_state_dict(best_state)

X_test = np.hstack([bert_probs_test, roberta_probs_test, electra_probs_test, distilbert_probs_test])
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
model.eval()
with torch.no_grad():
    final_preds = torch.argmax(model(X_test_t), dim=1).cpu().numpy()

submission_df = pd.DataFrame({'id': test_df['id'].astype(int), 'label': final_preds.astype(int)})
submission_df.to_csv('/kaggle/working/submission_mlp_honest_big_mlp.csv', index=False)

Epoch:  0  - Current score:  0.9045634920634921
Epoch:  1  - Current score:  0.9039351851851852
Epoch:  2  - Current score:  0.903968253968254
Epoch:  3  - Current score:  0.9049603174603175
Epoch:  4  - Current score:  0.9061177248677249
Epoch:  5  - Current score:  0.9043320105820106
Epoch:  6  - Current score:  0.9048941798941799
Epoch:  7  - Current score:  0.9034060846560846
Epoch:  8  - Current score:  0.9055886243386243
Epoch:  9  - Current score:  0.9041666666666667
Epoch:  10  - Current score:  0.9045634920634921
Epoch:  11  - Current score:  0.9030092592592592
Epoch:  12  - Current score:  0.9041335978835979
Epoch:  13  - Current score:  0.8999007936507937
Epoch:  14  - Current score:  0.9054563492063492
Epoch:  15  - Current score:  0.904431216931217
Epoch:  16  - Current score:  0.9060185185185186
Epoch:  17  - Current score:  0.9050595238095238
Epoch:  18  - Current score:  0.9055886243386243
Epoch:  19  - Current score:  0.9068121693121693
Epoch:  20  - Current score:  0.

KeyboardInterrupt: 

In [ ]:
print(f"Best HONEST holdout MLP score: {best_ho_score:.4f}")
model.load_state_dict(best_state)

X_test = np.hstack([bert_probs_test, roberta_probs_test, electra_probs_test, distilbert_probs_test])
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
model.eval()
with torch.no_grad():
    final_preds = torch.argmax(model(X_test_t), dim=1).cpu().numpy()

submission_df = pd.DataFrame({'id': test_df['id'].astype(int), 'label': final_preds.astype(int)})
submission_df.to_csv('/kaggle/working/submission_mlp_honest_big_mlp.csv', index=False)

Best HONEST holdout MLP score: 0.9081


In [ ]:
from itertools import product

avg_val = 0.20*bert_probs_val + 0.45*roberta_probs_val + 0.15*electra_probs_val + 0.20*distilbert_probs_val
ev_val = avg_val @ np.arange(5)

best = 0
best_t = None
for t1 in np.arange(0.3, 0.8, 0.02):
    for t2 in np.arange(t1+0.3, 1.8, 0.02):
        for t3 in np.arange(t2+0.3, 2.8, 0.02):
            for t4 in np.arange(t3+0.3, 3.8, 0.02):
                preds = np.zeros(len(ev_val), dtype=int)
                preds[ev_val >= t1] = 1
                preds[ev_val >= t2] = 2
                preds[ev_val >= t3] = 3
                preds[ev_val >= t4] = 4
                score = 1 - np.mean(np.abs(preds - y_val)) / 4
                if score > best:
                    best = score
                    best_t = (t1, t2, t3, t4)

print(f"Best threshold score: {best:.4f}, thresholds: {best_t}")

avg_test = 0.20*bert_probs_test + 0.45*roberta_probs_test + 0.15*electra_probs_test + 0.20*distilbert_probs_test
ev_test = avg_test @ np.arange(5)
t1, t2, t3, t4 = best_t
final_preds = np.zeros(len(ev_test), dtype=int)
final_preds[ev_test >= t1] = 1
final_preds[ev_test >= t2] = 2
final_preds[ev_test >= t3] = 3
final_preds[ev_test >= t4] = 4

submission_df = pd.DataFrame({'id': test_df['id'].astype(int), 'label': final_preds.astype(int)})
submission_df.to_csv('/kaggle/working/submission_thresholds.csv', index=False)

Best threshold score: 0.9068, thresholds: (np.float64(0.6400000000000003), np.float64(1.520000000000001), np.float64(2.4200000000000017), np.float64(3.440000000000002))
